# Axral Vocal AI — Kaggle One-cell pipeline
**入力は `HF_DATA_URL` だけ**。Hugging Face Dataset取得→音源監査→テスト→学習準備を1セルで実行します。
> 現状のリポジトリには実音声モデルの完全な学習CLIがまだないため、このセルは学習準備まで実行し、未実装の学習を偽装して起動しません。

In [ ]:
# ============================================================
# AXRAL VOCAL AI — ONE CELL KAGGLE PIPELINE
# 設定は HF_DATA_URL だけ。取得→文字コード対策→音源監査→pytest→学習データ準備
# ============================================================
import os, re, sys, json, subprocess, traceback
from pathlib import Path

# ★ここだけ変更：Hugging Face の Dataset URL（例 https://huggingface.co/datasets/OWNER/REPO）
HF_DATA_URL = "https://huggingface.co/datasets/SEKOIA2900/VO-US"
PROJECT = Path("/kaggle/working/Axral_vocal_AI")
DATA_ROOT = Path("/kaggle/working/hf_voicebanks")
OUT = Path("/kaggle/working/axral_run")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "pipeline.log"

def log(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as f: f.write(str(msg) + "\n")

def run(cmd, check=True):
    log("$ " + " ".join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=PROJECT, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, errors="replace")
    with LOG.open("a", encoding="utf-8") as f: f.write(p.stdout)
    print(p.stdout[-5000:])
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

LOG.write_text("", encoding="utf-8")
try:
    assert HF_DATA_URL.startswith("https://huggingface.co/"), "HF_DATA_URL must be a Hugging Face URL"
    assert PROJECT.exists(), f"Project not found: {PROJECT} (clone Axral_vocal_AI into /kaggle/working first)"
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"])
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

    # Dataset URL -> repository ID -> download/cache (private datasets use Kaggle Secrets HF_TOKEN)
    m = re.search(r"huggingface\.co/datasets/([^/?#]+/[^/?#]+)", HF_DATA_URL)
    if not m: raise ValueError("Dataset URL must look like huggingface.co/datasets/owner/name")
    dataset_id = m.group(1)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    from huggingface_hub import snapshot_download
    token = os.environ.get("HF_TOKEN")
    log(f"[1/5] Download dataset: {dataset_id}")
    snapshot_download(repo_id=dataset_id, repo_type="dataset", local_dir=str(DATA_ROOT),
                     token=token, resume_download=True)

    # Encoding safety: audit text is decoded by project reader (BOM/UTF-16/UTF-8/CP932).
    # Keep original voicebank files untouched; no destructive re-encoding.
    log("[2/5] Locate voicebank roots")
    banks = sorted({p.parent for p in DATA_ROOT.rglob("oto.ini")})
    log(f"oto.ini roots: {len(banks)}")
    if not banks: raise RuntimeError("No oto.ini found. Check HF dataset layout/download.")

    log("[3/5] Voicebank audit (JSON per bank)")
    audit_dir = OUT / "audits"; audit_dir.mkdir(exist_ok=True)
    for i, bank in enumerate(banks, 1):
        safe = re.sub(r"[^A-Za-z0-9._-]+", "_", bank.relative_to(DATA_ROOT).as_posix())[:100] or f"bank_{i}"
        target = audit_dir / f"{i:03d}_{safe}.json"
        run([sys.executable, "-m", "axral_vocal_ai.voicebank_audit_cli",
             str(bank), "--output", str(target)], check=False)
        log(f"audit {i}/{len(banks)}: {bank} -> {target}")

    log("[4/5] Tests")
    test_rc = run([sys.executable, "-m", "pytest", "-q"], check=False)
    log(f"pytest exit={test_rc}")

    log("[5/5] Training readiness")
    manifests = list(PROJECT.glob("**/*manifest*.json"))
    log(f"manifest candidates: {len(manifests)}")
    log("NOTE: Current repository has dataset validation/split/quality preparation, but no complete real-audio model training CLI yet.")
    log("Training is intentionally NOT started automatically until approved/licensed data manifest + actual trainer are present.")
    summary = {"dataset": dataset_id, "voicebank_roots": len(banks),
               "audit_files": len(list(audit_dir.glob("*.json"))), "pytest_exit": test_rc,
               "training": "not_implemented_end_to_end"}
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    log(json.dumps(summary, ensure_ascii=False))
except Exception:
    log(traceback.format_exc())
    raise
finally:
    log(f"LOG: {LOG}")
    log(f"OUTPUT: {OUT}")

